# 12 - A3: frozen A2 + the JEV API

**Question this notebook answers:** given A2's frozen per-clip score, does asking the JEV API
typed questions over a small, label-free evidence state add anything (experiment E5, A2
versus A3), and what would it cost to run this for real?

**A3 = frozen A2 + JEV.** A2 (a detector built on a frozen self-supervised learning (SSL)
speech model) already produces one score per clip in `runs/<a2 exp>/scores/dev.csv`.
A3 does not retrain anything: it builds a small JavaScript Object Notation (JSON) "evidence
state" per clip (A2's score/level plus a few acoustic summaries -- never the label, attack
id, speaker, path or clip id) and asks JEV (the "TypeSafe Jev API", a typed classifier
service) three fixed, versioned questions over that state:

1. **noul** -- "Does the evidence indicate synthetic speech?" (yes/no probabilities). This is
   the *only* question A3's own score file is derived from:
   `score = log(p(no) / p(yes))`, so higher = more bona fide, matching every other score file
   in this project.
2. **choice** -- bona fide / spoof / insufficient evidence.
3. **score** -- 0-3 evidence-consistency rating.

**Mode:** the real client, `TypesafeJevClient.ask`, is not implemented yet: it raises an
explicit `NotImplementedError` (see `audio_df/models/jev/client.py`'s TODO list). Every JEV call in
this notebook therefore goes through `DryRunJevClient`, a deterministic, clearly labelled
stand-in (`route="dry-run"`, `model_version="dry-run-0"`, `cost_usd=0.0`). **Nothing below is
a real JEV result.** A2 *does* exist: `configs/a3_jev.yaml: a2_experiment` points at its E1
benchmark run (`runs/E-20261001-e1-a2-raw`), so the evidence states below are built
from real A2 scores, real audio and A2's real dev threshold. If that run is missing (for
example on a fresh clone without the data), the notebook falls back to a small synthetic
fixture so every code path still runs.

In [17]:
# Work from the repo root, so config and manifest paths (all relative to it) resolve
# wherever the notebook is opened from (VS Code starts kernels in notebooks/).
import os
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO_ROOT)
print("working directory:", REPO_ROOT)

working directory: /Users/zuttome/LocalDocs/Learning/College/NUS/MTech-AIS/PS/GC2


In [18]:
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from audio_df.data import manifests as data
from audio_df.data.synthetic import build_synthetic_asvspoof_tree


def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(6):
        if (cur / "pyproject.toml").exists():
            return cur
        cur = cur.parent
    raise RuntimeError(f"could not find repo root (pyproject.toml) from {start}")


REPO_ROOT = _find_repo_root(Path.cwd())
print(f"REPO_ROOT = {REPO_ROOT}")

REPO_ROOT = /Users/zuttome/LocalDocs/Learning/College/NUS/MTech-AIS/PS/GC2


## Mode detection: real A2 dev scores, else a synthetic stand-in

Checks for A2's dev scores at `configs/a3_jev.yaml: a2_experiment`. With the E1 run and the
local ASVspoof audio present this runs in **REAL A2** mode (as in the stored output). Without
them it builds a synthetic fixture and a fake A2 score file instead.

In [19]:
_a3_cfg_path = REPO_ROOT / "configs" / "a3_jev.yaml"
_a3_cfg = yaml.safe_load(_a3_cfg_path.read_text())
_a2_dir = REPO_ROOT / _a3_cfg["a2_experiment"]

USE_REAL_A2 = (_a2_dir / "scores" / "dev.csv").exists()

_tmpdir = tempfile.TemporaryDirectory()
_tmp_root = Path(_tmpdir.name)

if USE_REAL_A2:
    MODE = "REAL A2"
    a2_scores_path = _a2_dir / "scores" / "dev.csv"
    dev_manifest_path = REPO_ROOT / "data" / "manifests" / "dev.csv"
    dev_manifest = pd.read_csv(dev_manifest_path)
else:
    MODE = "SYNTHETIC (no A2 run yet)"
    fixture_root = _tmp_root / "asvspoof_synth"
    synth_cfg = build_synthetic_asvspoof_tree(fixture_root, seed=1337, n_speakers_per_split=3)
    manifest = data.build_manifest(synth_cfg)
    dev_manifest = manifest[manifest["split"] == "dev"].reset_index(drop=True)

    # A fake A2: bonafide clips score positive, spoof negative, with a little noise -- just
    # enough for this notebook's dry-run demo to exercise every code path.
    rng = np.random.default_rng(0)
    fake_scores = pd.DataFrame(
        {
            "clip_id": dev_manifest["utt_id"],
            "label": dev_manifest["label"],
            "score": [
                (0.6 if lbl == "bonafide" else -0.6) + rng.normal(0, 0.1)
                for lbl in dev_manifest["label"]
            ],
            "attack_id": dev_manifest["attack_id"],
        }
    )
    a2_exp_dir = _tmp_root / "fake_a2_exp"
    (a2_exp_dir / "scores").mkdir(parents=True)
    fake_scores.to_csv(a2_exp_dir / "scores" / "dev.csv", index=False)
    a2_scores_path = a2_exp_dir / "scores" / "dev.csv"

    dev_manifest_path = _tmp_root / "dev_manifest.csv"
    dev_manifest.to_csv(dev_manifest_path, index=False)

print(f"MODE = {MODE}")
print(f"a2 dev scores: {a2_scores_path}")

MODE = REAL A2
a2 dev scores: /Users/zuttome/LocalDocs/Learning/College/NUS/MTech-AIS/PS/GC2/experiments/E-20261001-e1-a2-raw/scores/dev.csv


**Which detector threshold the evidence state is built against.** The level words
("strongly spoof" … "strongly bona fide") compare A2's score with A2's dev equal error rate
(EER) threshold.

- **REAL A2 mode:** the threshold is read from the A2 run's `metrics.json` by
  `audio_df.pipelines.a3.resolve_eer_threshold`, exactly as the runner does (−2.425 for
  the E1 run).
- **Synthetic mode:** there is no real metrics file, so the demo uses an explicit override of
  `0.0`, which matches the fake scores' sign convention. It is **not** a real A2 threshold.

In [20]:
from audio_df.pipelines.a3 import resolve_eer_threshold

demo_cfg = dict(_a3_cfg)
demo_cfg["thresholds"] = dict(_a3_cfg["thresholds"])
if USE_REAL_A2:
    threshold, threshold_source = resolve_eer_threshold(_a3_cfg, _a2_dir)
    print(f"A2 dev EER threshold = {threshold:.3f}  (from {threshold_source['source']})")
else:
    threshold = 0.0
    print("synthetic mode: demo eer_threshold override = 0.0 (not a real A2 threshold)")
demo_cfg["thresholds"]["detector"] = {**_a3_cfg["thresholds"]["detector"], "eer_threshold": threshold}

A2 dev EER threshold = -2.425  (from a2_metrics.testsets.dev.eer_threshold)


## One example evidence state, both variants

`audio_df.models.jev.state.build_state` turns one clip's A2 score + audio into a small,
JSON-serialisable, **label-free** dict. `"numbers"` keeps raw floats; `"words"` buckets
everything into threshold words (thresholds from `configs/a3_jev.yaml: thresholds`, applied
in code, never learned).

In [21]:
from audio_df.features.audio import load_audio
from audio_df.models.jev.state import build_state

a2_scores = pd.read_csv(a2_scores_path)
example_row = a2_scores.iloc[0]
example_clip_id = example_row["clip_id"]
example_audio_path = dev_manifest.loc[dev_manifest["utt_id"] == example_clip_id, "path"].iloc[0]
example_audio, _sr = load_audio(example_audio_path, target_sr=demo_cfg["sample_rate"])

state_numbers = build_state(
    example_clip_id, float(example_row["score"]), example_audio, demo_cfg, variant="numbers"
)
state_words = build_state(
    example_clip_id, float(example_row["score"]), example_audio, demo_cfg, variant="words"
)

print("numbers variant:")
print(json.dumps(state_numbers, indent=2))
print("\nwords variant:")
print(json.dumps(state_words, indent=2))

numbers variant:
{
  "variant": "numbers",
  "detector": {
    "score": -19.530536651611328,
    "level": "strongly spoof",
    "confidence": 1.0
  },
  "acoustic": {
    "high_band_energy": 0.014559115514501004,
    "silence_leading": 0.021798365122615803,
    "silence_trailing": 0.0,
    "silence_total": 0.021798365122615793,
    "duration_s": 1.468
  }
}

words variant:
{
  "variant": "words",
  "detector": {
    "level": "strongly spoof",
    "confidence": "high"
  },
  "acoustic": {
    "high_band_energy": "low",
    "silence_leading": "little",
    "silence_trailing": "little",
    "silence_total": "little",
    "duration_s": "short"
  }
}


**What is *not* in either dict:** the clip id, the file path, the label, the attack id and the
speaker. The level word ("strongly spoof", "bona fide", …) *is* there on purpose: it is A2's
opinion, which A3 is meant to see, not the true answer.

`tests/test_jev_state.py::test_no_label_leakage` checks the property that matters: for every
level word and both variants, two different clip ids with the same score and audio give
byte-identical JSON. So nothing about the clip's identity can be in the state, and no key
anywhere names an identity or label field. `test_state_keys_are_exactly_the_documented_ones`
fixes the allowed keys, so a new field cannot slip in unnoticed.

## The three questions

Versioned text (`audio_df.models.jev.questions.QUESTION_TEXT_VERSION`), so a wording change
invalidates cached answers rather than silently mixing them in with answers to the old
wording.

In [22]:
from audio_df.models.jev.questions import CHOICE_QUESTION, NOUL_QUESTION, QUESTION_SETS, SCORE_QUESTION

for q in (NOUL_QUESTION, CHOICE_QUESTION, SCORE_QUESTION):
    print(f"[{q.id:>6}] ({q.question_type}, {q.version}) {q.text}")
    if q.options:
        print(f"          options: {q.options}")
    if q.scale:
        print(f"          scale:   {q.scale}")

print("\nquestion sets:", {k: [q.id for q in v] for k, v in QUESTION_SETS.items()})

[  noul] (noul, v1) Does the evidence indicate synthetic speech?
[choice] (choice, v1) Given the evidence, is this clip bona fide, spoof, or is the evidence insufficient?
          options: ('bonafide', 'spoof', 'insufficient_evidence')
[ score] (score, v1) On a scale of 0 (strongly inconsistent) to 3 (strongly consistent), how consistent is this evidence with genuine (bona fide) speech?
          scale:   (0, 3)

question sets: {'noul_only': ['noul'], 'all3': ['noul', 'choice', 'score']}


## A dry run on 5 clips

Every call below goes through `DryRunJevClient` -- deterministic, `route="dry-run"`,
`cost_usd=0.0`. This exercises the full runner (`audio_df.pipelines.a3.run_a3`): the
cache, the state builder, both score files, `answers.csv`, and the cost/latency summary.

In [23]:
from audio_df.pipelines.a3 import run_a3

demo_cfg["a2_experiment"] = str(a2_scores_path.parent.parent)
demo_cfg["subset"] = {
    "manifest": str(dev_manifest_path),
    "n": 5,
    "seed": 1337,
    "strata": ["label", "attack_id"],
}
demo_cfg["cache_dir"] = str(_tmp_root / "jev_cache")

exp_dir = run_a3(
    demo_cfg,
    state_variant="numbers",
    question_set="noul_only",
    client_kind="dry-run",
    root=str(_tmp_root / "experiments"),
    lock_path=str(_tmp_root / "lock"),
    status_fn=lambda: {
        "pressure_level": 1,
        "pressure_name": "normal",
        "swap_used_gb": 0.1,
        "swap_total_gb": 1.0,
        "rss_mb": 1.0,
    },
)
print(f"exp_dir = {exp_dir}")
pd.read_csv(exp_dir / "scores" / "jev_dev500.csv")

{
  "cache_calls": 5,
  "cache_hits": 0,
  "cost_mean_usd": 0.0,
  "cost_total_usd": 0.0,
  "latency_median_s": 9.625015081837773e-06,
  "latency_p90_s": 1.8833013018593193e-05,
  "n_clips": 5,
  "n_questions": 5
}
Next: uv run python scripts/score_experiment.py /var/folders/hv/lwt64sl51d78tksf3db976yh0000gn/T/tmpd0a47pcu/experiments/E-20261005-e5-a3-raw-numbers-noul-dryrun --threshold-from <a2 dev exp dir, if a3 has no fresh dev threshold of its own>
Then: uv run python scripts/make_tables.py
exp_dir = /var/folders/hv/lwt64sl51d78tksf3db976yh0000gn/T/tmpd0a47pcu/experiments/E-20261005-e5-a3-raw-numbers-noul-dryrun


,clip_id,label,score,attack_id
0,LA_D_4859938,spoof,-2.197225,A03
1,LA_D_6320458,spoof,-2.197225,A05
2,LA_D_6851421,spoof,-2.197225,A01
3,LA_D_9028753,spoof,-2.197225,A04
4,LA_D_9039099,spoof,-2.197225,A02


In [24]:
summary = json.loads((exp_dir / "cost_summary.json").read_text())
print(json.dumps(summary, indent=2, sort_keys=True))

{
  "cache_calls": 5,
  "cache_hits": 0,
  "cost_mean_usd": 0.0,
  "cost_total_usd": 0.0,
  "latency_median_s": 9.625015081837773e-06,
  "latency_p90_s": 1.8833013018593193e-05,
  "n_clips": 5,
  "n_questions": 5
}


### What this shows

All 5 clips in the subset got a score, and all 5 scores are identical (-2.197). That is
expected from the stand-in: `DryRunJevClient` answers from A2's level word alone, and all five
sampled dev clips are spoofs (attacks A01-A05) at the "strongly spoof" level, which maps to
p(yes) = 0.9 and a score of log(0.1 / 0.9) ≈ -2.197. `cost_summary.json` reports 0 dollars
and microsecond latencies, because the stand-in is free and in-process. **None of this is
evidence about the real JEV API's answers, latency or cost.**

### What it means

The plumbing (state → cache → client → score file → cost summary) works end to end on real
A2 scores in REAL A2 mode. That is the only claim this cell supports.

## The plan once `TypesafeJevClient.ask` is implemented

1. **Measure first, on <= 20 real calls.** `run_a3`'s budget guard already refuses
   `--client typesafe` runs above `n=20` until a real cost-per-call is known (see
   `audio_df/pipelines/a3.py`'s `_TYPESAFE_FIRST_RUN_MAX_N`). Run all four
   `(state, question_set)` combinations once each at `n<=20`, read the printed
   `cost_summary.json` (`cost_mean_usd`, `latency_median_s`, `latency_p90_s`), and only then
   decide the real subset size (the config default is 500, per E5's plan) against a real
   budget.
2. **Then scale up** with `--n 500` (or whatever the measured budget affords) and `--resume`
   if a run is interrupted -- the cache means resuming never re-pays for an already-answered
   question.
3. **Compare A2 vs A3** on the identical clip set (`scores/a2_dev500.csv` vs
   `scores/jev_dev500.csv`, written by the same run) via
   `scripts/score_experiment.py` + `scripts/make_tables.py`'s E5 table.

### The four variant commands

(`state` x `question_set`; run each once at `n<=20` first per the plan above)

In [25]:
for state in _a3_cfg["variants"]["state"]:
    for question_set in _a3_cfg["variants"]["question_set"]:
        print(
            "uv run adf --config configs/a3_jev.yaml jev "
            f"--state {state} --questions {question_set} --client dry-run --n 20"
        )

uv run python -m audio_df.pipelines.run_a3 run --state numbers --questions noul_only --client dry-run --n 20
uv run python -m audio_df.pipelines.run_a3 run --state numbers --questions all3 --client dry-run --n 20
uv run python -m audio_df.pipelines.run_a3 run --state words --questions noul_only --client dry-run --n 20
uv run python -m audio_df.pipelines.run_a3 run --state words --questions all3 --client dry-run --n 20


## After a real run: A2 vs A3, and answer distributions

Not executable yet: A2 exists, but no real JEV run does, because `TypesafeJevClient.ask`
still raises `NotImplementedError`. This is the analysis to run once it is implemented:

```python
a2 = pd.read_csv(exp_dir / "scores" / "a2_dev500.csv")
a3 = pd.read_csv(exp_dir / "scores" / "jev_dev500.csv")
merged = a2.merge(a3, on=["clip_id", "label"], suffixes=("_a2", "_a3"))
merged[["score_a2", "score_a3"]].corr()

answers = pd.read_csv(exp_dir / "answers.csv")
answers.groupby("question_id")["result"].apply(lambda s: s.value_counts().head())
```

Then: `uv run python scripts/score_experiment.py <exp_dir>` and
`uv run python scripts/make_tables.py` to regenerate the E5 (A2-vs-A3) table.

## Try it by hand: inputs for the JEV playground

Until `TypesafeJevClient.ask` is implemented, A3's idea can be tested by pasting its inputs
into the JEV web playground by hand. This section builds its **own** evidence state; it does
not use `build_state`.

**What is left out, on purpose.** Every property that notebooks 01 and 02 tie to how the
dataset was collected rather than to how the fake was made: leading, trailing and total
silence, loudness (root-mean-square level and LFCC c0), duration, sample rate and codec.
Sending silence or duration would hand JEV the same shortcut A2 can use, with no way for JEV
to know it is one.

**What goes in: speech-only acoustic evidence.** Each clip's leading and trailing silence is
cut first (`trim_silence`), then:

- four spectral summaries from notebook 02: **bandwidth** and **rolloff** (the two largest
  spectral, non-silence real/fake differences there), **centroid** and **flatness**;
- four shape coefficients of the official baseline's linear-frequency cepstral coefficients
  (LFCC), averaged over frames, chosen from notebook 03's per-coefficient comparison:
  **c1** (spectral tilt), **c5** (envelope), **c16** and **c18** (fine detail).

A raw number like "c5 = 0.78" means nothing to JEV on its own, so each value comes with two
**percentiles**: where the clip sits among a reference sample of **real** training clips and
among **fake** training clips (A01–A06, train split only, so nothing from dev or In-the-Wild).
The reference uses the same speech-only measurement.

**Two versions of every clip**, to test whether JEV reads the evidence or copies A2:

- **A) evidence only** (no detector at all);
- **B) evidence + A2**: A2's score, its dev equal error rate (EER) threshold (−2.425) and which
  side the score falls on. Version B does not use `build_state`'s "strongly …" level words,
  whose margins are not calibrated on A2's score scale.

**Six clips:** two obvious dev clips (one real, one fake), the two dev clips closest to A2's
threshold, one real In-the-Wild clip that A2 wrongly calls fake, one In-the-Wild fake. The
answer key is printed two cells down; look at it only after asking JEV.

Set `PLAYGROUND_QUESTIONS` to `"noul_only"` or `"all3"`, and `N_REF` (reference clips per class;
the reference takes about a minute at 150).

In [26]:
import librosa

from audio_df.features.audio import load_audio, trim_silence
from audio_df.pipelines.a3 import resolve_eer_threshold
from audio_df.models.jev.questions import QUESTION_SETS
from audio_df.features.lfcc_b02 import extract_lfcc_b02

PLAYGROUND_QUESTIONS = "all3"   # "noul_only" or "all3"
N_REF = 150                     # reference clips per class, from the train split
SR = 16000

A2_DEV_RUN = REPO_ROOT / "experiments" / "E-20261001-e1-a2-raw"
A2_ITW_RUN = REPO_ROOT / "experiments" / "E-20261001-e3-a2-raw"

pg_cfg = dict(_a3_cfg)
pg_cfg["thresholds"] = {**_a3_cfg["thresholds"],
                        "detector": {**_a3_cfg["thresholds"]["detector"], "eer_threshold": None}}
pg_threshold, pg_threshold_source = resolve_eer_threshold(pg_cfg, A2_DEV_RUN)
print(f"A2 dev EER threshold = {pg_threshold:.3f}  ({pg_threshold_source['source']})")

# Speech-only evidence. Deliberately absent: silence, loudness (rms, LFCC c0), duration,
# sample rate, codec -- the dataset-collection properties found in notebooks 01 and 02.
LFCC_KEEP = {1: "lfcc_c1_tilt", 5: "lfcc_c5_envelope", 16: "lfcc_c16_fine_detail", 18: "lfcc_c18_fine_detail"}

def speech_features(x):
    x = trim_silence(x, SR)
    kw = dict(y=x, sr=SR, n_fft=512, hop_length=160)
    feats = {
        "spectral_bandwidth_hz": float(librosa.feature.spectral_bandwidth(**kw).mean()),
        "spectral_rolloff_hz": float(librosa.feature.spectral_rolloff(**kw).mean()),
        "spectral_centroid_hz": float(librosa.feature.spectral_centroid(**kw).mean()),
        "spectral_flatness": float(librosa.feature.spectral_flatness(y=x, n_fft=512, hop_length=160).mean()),
    }
    static = extract_lfcc_b02(x, sr=SR)[:, :20].mean(axis=0)
    feats.update({name: float(static[k]) for k, name in LFCC_KEEP.items()})
    return feats

# Reference: N_REF real and N_REF fake TRAIN clips (seeded), measured the same way.
train = pd.read_csv(REPO_ROOT / "data" / "manifests" / "train.csv")
ref_rows = pd.concat([train[train.label == lab].sample(N_REF, random_state=1337)
                      for lab in ("bonafide", "spoof")])
ref = pd.DataFrame([{"label": r.label, **speech_features(load_audio(REPO_ROOT / r.path, target_sr=SR)[0])}
                    for r in ref_rows.itertuples(index=False)])
print(f"reference: {N_REF} real + {N_REF} fake train clips")
print(ref.groupby("label").median().T.round(3))

def percentile(value, sample):
    return int(round(100 * (np.asarray(sample) < value).mean()))

def evidence(feats):
    out = {"how_to_read": ("measured on speech only (leading/trailing silence removed); "
                           "pct_of_real / pct_of_fake = percent of real / fake reference "
                           "training clips with a lower value")}
    for name, v in feats.items():
        out[name] = {"value": round(v, 4),
                     "pct_of_real": percentile(v, ref.loc[ref.label == "bonafide", name]),
                     "pct_of_fake": percentile(v, ref.loc[ref.label == "spoof", name])}
    return out

dev_scores = pd.read_csv(A2_DEV_RUN / "scores" / "dev.csv")
itw_scores = pd.read_csv(A2_ITW_RUN / "scores" / "itw.csv")
dev_paths = pd.read_csv(REPO_ROOT / "data" / "manifests" / "dev.csv").set_index("utt_id")["path"]
itw_paths = pd.read_csv(REPO_ROOT / "data" / "manifests" / "itw.csv").set_index("utt_id")["path"]

rng_pg = np.random.default_rng(1337)
def _pick(df):
    return df.iloc[int(rng_pg.integers(len(df)))]

dev_scores["gap"] = (dev_scores["score"] - pg_threshold).abs()
near = dev_scores.nsmallest(2, "gap")
picked = [
    ("dev, clearly real",  _pick(dev_scores[(dev_scores.label == "bonafide") & (dev_scores.score > pg_threshold + 10)]), dev_paths),
    ("dev, clearly fake",  _pick(dev_scores[(dev_scores.label == "spoof") & (dev_scores.score < pg_threshold - 10)]), dev_paths),
    ("dev, near the threshold (1)", near.iloc[0], dev_paths),
    ("dev, near the threshold (2)", near.iloc[1], dev_paths),
    ("In-the-Wild real, A2 calls it fake", _pick(itw_scores[(itw_scores.label == "bonafide") & (itw_scores.score < pg_threshold)]), itw_paths),
    ("In-the-Wild fake", _pick(itw_scores[itw_scores.label == "spoof"]), itw_paths),
]

playground = []
for n, (why, row, paths) in enumerate(picked, start=1):
    audio, _ = load_audio(REPO_ROOT / paths[row["clip_id"]], target_sr=SR)
    ev = evidence(speech_features(audio))
    a2 = {"score": round(float(row["score"]), 3), "threshold": round(pg_threshold, 3),
          "says": "bona fide" if row["score"] > pg_threshold else "spoof",
          "note": "A2 is a pre-trained-speech detector; score above threshold = bona fide"}
    playground.append({"n": n, "why": why, "clip_id": row["clip_id"], "label": row["label"],
                       "a2_score": float(row["score"]),
                       "states": {"A_evidence_only": {"speech_evidence": ev},
                                  "B_evidence_plus_A2": {"detector_A2": a2, "speech_evidence": ev}}})

print("\nQuestions (same for every clip and version):")
for k, q in enumerate(QUESTION_SETS[PLAYGROUND_QUESTIONS], start=1):
    kind = {"noul": "Noul (yes/no probability)", "choice": "Choice", "score": "Score"}[q.question_type]
    extra = f"  | options: {', '.join(q.options)}" if q.options else ""
    extra = f"  | scale: {q.scale[0]} to {q.scale[1]}" if q.scale else extra
    print(f"  Q{k} [{kind}] {q.text}{extra}")
for item in playground:
    for version, state in item["states"].items():
        print("=" * 78)
        print(f"CLIP {item['n']} of {len(playground)}, version {version}: paste this JSON as the input")
        print(json.dumps(state, indent=2))
print("=" * 78)

A2 dev EER threshold = -2.425  (a2_metrics.testsets.dev.eer_threshold)
reference: 150 real + 150 fake train clips
label                  bonafide     spoof
spectral_bandwidth_hz  1444.754  1351.250
spectral_rolloff_hz    3072.693  2882.095
spectral_centroid_hz   1721.384  1604.250
spectral_flatness         0.040     0.033
lfcc_c1_tilt              2.212     2.433
lfcc_c5_envelope          0.386     0.495
lfcc_c16_fine_detail     -0.021    -0.041
lfcc_c18_fine_detail     -0.031    -0.046

Questions (same for every clip and version):
  Q1 [Noul (yes/no probability)] Does the evidence indicate synthetic speech?
  Q2 [Choice] Given the evidence, is this clip bona fide, spoof, or is the evidence insufficient?  | options: bonafide, spoof, insufficient_evidence
  Q3 [Score] On a scale of 0 (strongly inconsistent) to 3 (strongly consistent), how consistent is this evidence with genuine (bona fide) speech?  | scale: 0 to 3
CLIP 1 of 6, version A_evidence_only: paste this JSON as the input
{
  "

### Reading the reference, then the answer key (look at the key after asking JEV)

**Reference medians (measured above, 150 real and 150 fake train clips, speech only).** Every
evidence feature differs between the classes in the same direction as in notebooks 02 and 03,
so removing silence does not remove these differences: fakes have lower bandwidth (1,351 vs
1,445 Hz), rolloff (2,882 vs 3,073 Hz), centroid (1,604 vs 1,721 Hz) and flatness (0.033 vs
0.040), higher c1 (2.433 vs 2.212) and c5 (0.495 vs 0.386), and lower c16 (-0.041 vs -0.021)
and c18 (-0.046 vs -0.031). The gaps are small next to the spread within each class (clip 1,
a real clip, sits at the 79th percentile of fakes for bandwidth), so no single feature
decides a clip.

**Answer key.** What each clip really is, and what A2 decided at its dev threshold. In the
stored run A2 is right on 4 of the 6 clips; it is wrong on clip 4 (a dev fake scoring -2.42,
just above the -2.425 threshold) and clip 5 (an In-the-Wild real clip scoring -4.25).

In [27]:
for item in playground:
    a2_says = "real" if item["a2_score"] > pg_threshold else "fake"
    truth = "real" if item["label"] == "bonafide" else "fake"
    print(f"clip {item['n']}: {item['why']:<36} truth={truth:<4}  A2 score={item['a2_score']:+7.2f}"
          f"  A2 says {a2_says:<4} ({'right' if a2_says == truth else 'WRONG'})  [{item['clip_id']}]")

clip 1: dev, clearly real                    truth=real  A2 score= +11.73  A2 says real (right)  [LA_D_6016215]
clip 2: dev, clearly fake                    truth=fake  A2 score= -20.57  A2 says fake (right)  [LA_D_8879769]
clip 3: dev, near the threshold (1)          truth=fake  A2 score=  -2.43  A2 says fake (right)  [LA_D_3754369]
clip 4: dev, near the threshold (2)          truth=fake  A2 score=  -2.42  A2 says real (WRONG)  [LA_D_8149486]
clip 5: In-the-Wild real, A2 calls it fake   truth=real  A2 score=  -4.25  A2 says fake (WRONG)  [ITW_30737]
clip 6: In-the-Wild fake                     truth=fake  A2 score= -10.79  A2 says fake (right)  [ITW_15298]


### Record JEV's answers

Type in the **"yes" probability** JEV returned for the Noul question ("Does the evidence
indicate synthetic speech?"), for each clip and each version, then run the cell.

- **A3's score** is the log-odds of "no": $\log\frac{p(\text{no})}{p(\text{yes})}$
  ([`noul_to_a3_score`](../src/audio_df/models/jev/questions.py)). Higher means more real, like every
  other score in the project; positive = JEV leans "real".
- **Compare A with B.** If B always matches A2 and A does not, JEV is following A2's verdict.
  If A is already right on a clip A2 gets wrong, the speech evidence carries something A2 missed.

These hand-typed numbers are a playground check only, not an experiment result. Nothing here
is written to `experiments/`.


In [28]:
from audio_df.models.jev.questions import noul_to_a3_score

JEV_P_YES = {   # fill in, e.g. 1: 0.12
    "A_evidence_only":    {1: None, 2: None, 3: None, 4: None, 5: None, 6: None},
    "B_evidence_plus_A2": {1: None, 2: None, 3: None, 4: None, 5: None, 6: None},
}

def _says(a3):
    return "real" if a3 > 0 else "fake"

rows = []
for item in playground:
    truth = "real" if item["label"] == "bonafide" else "fake"
    row = {"clip": item["n"], "why": item["why"], "truth": truth,
           "A2 says": "real" if item["a2_score"] > pg_threshold else "fake"}
    for version, answers in JEV_P_YES.items():
        p_yes = answers.get(item["n"])
        if p_yes is not None:
            a3 = noul_to_a3_score({"yes": p_yes, "no": 1.0 - p_yes})
            row[f"{version[0]}: p(yes)"] = p_yes
            row[f"{version[0]}: A3 says"] = _says(a3)
    rows.append(row)
filled = any(v is not None for answers in JEV_P_YES.values() for v in answers.values())
pd.DataFrame(rows) if filled else print("Fill in JEV_P_YES above with JEV's answers, then re-run this cell.")


Fill in JEV_P_YES above with JEV's answers, then re-run this cell.


In [29]:
_tmpdir.cleanup()